In [1]:
import torch
import sys

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Python:", sys.version)

GPU: NVIDIA A100-PCIE-40GB
PyTorch: 2.12.0+cu130
CUDA: 13.0
Python: 3.12.15 (main, Oct 13 2026, 16:52:04) [GCC 15.3.0]


In [2]:
# Environment + reproducibility
import os, subprocess, random, math, json, time, urllib.request
from pathlib import Path
from dataclasses import dataclass, asdict

# Match the verified Phase-3 continuation environment.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Preserve the same TF32 behavior used by the prior notebook.
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("CUBLAS_WORKSPACE_CONFIG:", os.environ.get("CUBLAS_WORKSPACE_CONFIG"))
print("PYTHONHASHSEED:", os.environ.get("PYTHONHASHSEED"))
print("Deterministic algorithms:", torch.are_deterministic_algorithms_enabled())
print("cuDNN deterministic:", torch.backends.cudnn.deterministic)
print("cuDNN benchmark:", torch.backends.cudnn.benchmark)

assert torch.__version__ == "2.12.0+cu130", torch.__version__
assert torch.version.cuda == "13.0", torch.version.cuda
if DEVICE == "cuda":
    assert torch.cuda.get_device_name(0) == "NVIDIA A100-PCIE-40GB"


PyTorch: 2.12.0+cu130
PyTorch CUDA: 13.0
Device: cuda
GPU: NVIDIA A100-PCIE-40GB
CUBLAS_WORKSPACE_CONFIG: :4096:8
PYTHONHASHSEED: 0
Deterministic algorithms: True
cuDNN deterministic: True
cuDNN benchmark: False


In [3]:
# Configuration: clean 10M Phase-3 setup
PROJECT_ID = '10m_diagonal_attention_mechanism_test'
ROOT = Path('/workspace') if Path('/workspace').exists() else Path('.')
OUT_DIR = ROOT / PROJECT_ID
CKPT_DIR = OUT_DIR / 'results'
DATA_DIR = OUT_DIR / 'data'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

N_LAYER    = 6
N_EMBD     = 384
N_HEAD     = 6
BLOCK_SIZE = 256
DROPOUT    = 0.2

BATCH_SIZE    = 64
MAX_ITERS     = 3000
EVAL_INTERVAL = 100
EVAL_BATCHES  = 50
LEARNING_RATE = 3e-4
MIN_LR        = 3e-5
WARMUP_ITERS  = 100
WEIGHT_DECAY  = 0.1
BETA1, BETA2  = 0.9, 0.95
GRAD_CLIP     = 1.0
FIXED_VAL_SEED = 99991

BASE_SEED = 1337
FOLLOWUP_SEEDS = [42, 123]
MATCHED_ATTNONLY_SEED1337 = 1.60845
MATCHED_LOCALMIXER_SEED1337 = 2.490946664810181
MATCHED_FFNONLY_SEED1337 = 3.3488304662704467
HISTORICAL_PHASE3_ATTNONLY_MEAN = 1.580
HISTORICAL_PHASE3_FFNONLY_MEAN = 3.350

print('Output:', OUT_DIR)
print('Matched prior AttnOnly seed-1337 best val:', MATCHED_ATTNONLY_SEED1337)


Output: /workspace/10m_diagonal_attention_mechanism_test
Matched prior AttnOnly seed-1337 best val: 1.60845


In [4]:
# Cell 3 - Dataset (TinyShakespeare, character-level — identical to original)
import urllib.request

DATA_DIR = OUT_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)
txt_path = DATA_DIR / 'input.txt'

if not txt_path.exists():
    print('Downloading TinyShakespeare...')
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        txt_path
    )
text = txt_path.read_text()

chars = sorted(set(text))
VOCAB_SIZE = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f'Vocab size: {VOCAB_SIZE}')
print(f'Train tokens: {len(train_data):,}')
print(f'Val tokens:   {len(val_data):,}')


Vocab size: 65
Train tokens: 1,003,854
Val tokens:   111,540


In [5]:
# Cell 4 - Batcher with FIXED validation batches (critical architecture detail)
#
# This is the single most important fix from the original nanoGPT pilot:
# random val batches at every eval introduce noise that masks slow learners
# like AttnOnly. Fixed batches (seed 99,991) ensure identical evaluation
# data across every step and every seed, so seed-to-seed variance reflects
# the model, not the evaluation sample.

class SequenceBatcher:
    def __init__(self, train_data, val_data, block_size, batch_size, device,
                 fixed_val_seed=FIXED_VAL_SEED, n_fixed_val_batches=50):
        self.train_data = train_data
        self.val_data   = val_data
        self.block_size = block_size
        self.batch_size = batch_size
        self.device     = device

        # Build fixed val batches ONCE, using an isolated generator so it
        # never interacts with the global / per-experiment seed state.
        g = torch.Generator().manual_seed(fixed_val_seed)
        self.fixed_val_batches = []
        for _ in range(n_fixed_val_batches):
            ix = torch.randint(len(val_data) - block_size, (batch_size,), generator=g)
            xb = torch.stack([val_data[i:i+block_size]     for i in ix]).to(device)
            yb = torch.stack([val_data[i+1:i+block_size+1] for i in ix]).to(device)
            self.fixed_val_batches.append((xb, yb))

    def sample(self, split):
        d = self.train_data if split == 'train' else self.val_data
        ix = torch.randint(len(d) - self.block_size, (self.batch_size,))
        xb = torch.stack([d[i:i+self.block_size]     for i in ix]).to(self.device)
        yb = torch.stack([d[i+1:i+self.block_size+1] for i in ix]).to(self.device)
        return xb, yb

# Build ONCE — shared across all seeds in this notebook, exactly as in
# the original ResidualGPT run, so every config sees identical val data.
BATCHER = SequenceBatcher(train_data, val_data, BLOCK_SIZE, BATCH_SIZE, DEVICE)
print(f'Fixed val batches built: {len(BATCHER.fixed_val_batches)} batches, seed=99991')


Fixed val batches built: 50 batches, seed=99991


In [6]:
# Cell 5 - ResidualGPT model
# Critical correctness requirements baked in here:
#   (a) NO runtime gain multiplier in forward() — gain is applied to WEIGHTS
#       at init time only, via apply_output_gains(), never as a forward-pass
#       multiplier. (That bug was caught and fixed in the original project —
#       see paper Section 5.1.)
#   (b) Weight tying between token_embed and lm_head.
#   (c) attention_skip / mlp_skip flags control residual connections.

@dataclass
class ResidualSwitch:
    use_attention_skip: bool = True
    use_mlp_skip:       bool = True
    attention_gain:     float = 1.0
    mlp_gain:           float = 1.0


class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        assert N_EMBD % N_HEAD == 0
        self.qkv  = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)
        self.out  = nn.Linear(N_EMBD, N_EMBD, bias=False)
        self.drop = nn.Dropout(DROPOUT)
        self.n_head = N_HEAD
        self.head_dim = N_EMBD // N_HEAD

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.drop.p if self.training else 0.0, is_causal=True
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.out(y))


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(N_EMBD, 4 * N_EMBD, bias=False)
        self.fc2  = nn.Linear(4 * N_EMBD, N_EMBD, bias=False)
        self.drop = nn.Dropout(DROPOUT)

    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))


class Block(nn.Module):
    def __init__(self, switch: ResidualSwitch):
        super().__init__()
        self.switch    = switch
        self.ln1       = nn.LayerNorm(N_EMBD)
        self.ln2       = nn.LayerNorm(N_EMBD)
        self.attn      = CausalSelfAttention()
        self.mlp       = MLP()

    def forward(self, x):
        # NO runtime multiplier here. Gain is purely a weight-init-time effect.
        a = self.attn(self.ln1(x))
        x = x + a if self.switch.use_attention_skip else a
        m = self.mlp(self.ln2(x))
        x = x + m if self.switch.use_mlp_skip else m
        return x


class ResidualGPT(nn.Module):
    def __init__(self, switch: ResidualSwitch):
        super().__init__()
        self.switch = switch
        self.token_embed = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_embed   = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.drop        = nn.Dropout(DROPOUT)
        self.blocks      = nn.ModuleList([Block(switch) for _ in range(N_LAYER)])
        self.ln_f        = nn.LayerNorm(N_EMBD)
        self.lm_head     = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)

        # Weight tying
        self.lm_head.weight = self.token_embed.weight

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def apply_output_gains(self):
        """Weight-level gain application, called ONCE before optimizer construction.
        With gain=1.0 (this run's config) this is a no-op, included only for
        architectural parity with the original implementation."""
        if self.switch.mlp_gain != 1.0:
            with torch.no_grad():
                for block in self.blocks:
                    block.mlp.fc2.weight.mul_(self.switch.mlp_gain)
        if self.switch.attention_gain != 1.0:
            with torch.no_grad():
                for block in self.blocks:
                    block.attn.out.weight.mul_(self.switch.attention_gain)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.token_embed(idx) + self.pos_embed(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def param_count(self):
        return sum(p.numel() for p in self.parameters())

print('Model class defined')


Model class defined


In [7]:
class LocalMixer(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(N_EMBD, 2 * N_EMBD, bias=False)
        self.fc2 = nn.Linear(2 * N_EMBD, N_EMBD, bias=False)
        self.drop = nn.Dropout(DROPOUT)

    def forward(self, x):
        # x: (B,T,D); nn.Linear acts independently at each token position.
        return self.drop(self.fc2(F.gelu(self.fc1(x))))


class LocalMixerBlock(nn.Module):
    def __init__(self, switch: ResidualSwitch):
        super().__init__()
        self.switch = switch
        self.ln1 = nn.LayerNorm(N_EMBD)
        self.ln2 = nn.LayerNorm(N_EMBD)
        self.mixer = LocalMixer()
        self.mlp = MLP()

    def forward(self, x):
        a = self.mixer(self.ln1(x))
        x = x + a if self.switch.use_attention_skip else a
        m = self.mlp(self.ln2(x))
        x = x + m if self.switch.use_mlp_skip else m
        return x


class LocalMixerGPT(nn.Module):
    def __init__(self, switch: ResidualSwitch):
        super().__init__()
        self.switch = switch
        self.token_embed = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_embed = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.drop = nn.Dropout(DROPOUT)
        self.blocks = nn.ModuleList([LocalMixerBlock(switch) for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.lm_head = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)
        self.lm_head.weight = self.token_embed.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def apply_output_gains(self):
        if self.switch.mlp_gain != 1.0:
            with torch.no_grad():
                for block in self.blocks:
                    block.mlp.fc2.weight.mul_(self.switch.mlp_gain)
        if self.switch.attention_gain != 1.0:
            # In the intervention, attention_gain maps to the output projection
            # of the replacement mixer. Gain=1.0 in this experiment.
            with torch.no_grad():
                for block in self.blocks:
                    block.mixer.fc2.weight.mul_(self.switch.attention_gain)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.token_embed(idx) + self.pos_embed(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def param_count(self):
        return sum(p.numel() for p in self.parameters())

attn_params = sum(p.numel() for p in CausalSelfAttention().parameters())
mixer_params = sum(p.numel() for p in LocalMixer().parameters())
print(f'Attention params: {attn_params:,}')
print(f'LocalMixer params: {mixer_params:,}')
assert attn_params == mixer_params, 'Parameter budgets do not match.'
print('PASS: attention and LocalMixer parameter counts match exactly.')


Attention params: 589,824
LocalMixer params: 589,824
PASS: attention and LocalMixer parameter counts match exactly.


In [8]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)


def learning_rate_at(step, max_iters=MAX_ITERS, base_lr=LEARNING_RATE,
                     min_lr=MIN_LR, warmup_iters=WARMUP_ITERS):
    if step < warmup_iters:
        return base_lr * (step + 1) / max(1, warmup_iters)
    if step >= max_iters:
        return min_lr
    progress = (step - warmup_iters) / max(1, max_iters - warmup_iters)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_lr + cosine * (base_lr - min_lr)


@torch.no_grad()
def estimate_val_loss(model, n_batches=EVAL_BATCHES):
    model.eval()
    losses = []
    for xb, yb in BATCHER.fixed_val_batches[:n_batches]:
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


@torch.no_grad()
def estimate_train_loss(model, n_batches=EVAL_BATCHES):
    model.eval()
    losses = []
    for _ in range(n_batches):
        xb, yb = BATCHER.sample('train')
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def make_optimizer(model):
    decay, no_decay = [], []
    for _, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (decay if p.ndim >= 2 else no_decay).append(p)
    return torch.optim.AdamW(
        [{'params': decay, 'weight_decay': WEIGHT_DECAY},
         {'params': no_decay, 'weight_decay': 0.0}],
        lr=LEARNING_RATE, betas=(BETA1, BETA2)
    )


def result_path(condition, seed):
    return CKPT_DIR / f'{condition}_seed{seed}.json'


def save_result(result):
    path = result_path(result['condition'], result['seed'])
    with open(path, 'w') as f:
        json.dump(result, f, indent=2)
    print('Saved:', path)


def load_result(condition, seed):
    path = result_path(condition, seed)
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None


def train_one_run(condition, seed, model_cls, switch, max_iters=MAX_ITERS,
                  eval_interval=EVAL_INTERVAL, eval_batches=EVAL_BATCHES, verbose=True):
    set_seed(seed)
    model = model_cls(switch).to(DEVICE)
    model.apply_output_gains()
    optimizer = make_optimizer(model)

    history, best_val, best_step = [], float('inf'), -1
    started = time.time()

    val0 = estimate_val_loss(model, eval_batches)
    train0 = estimate_train_loss(model, eval_batches)
    history.append({'step': 0, 'lr': learning_rate_at(0, max_iters),
                    'train_loss': train0, 'val_loss': val0})
    best_val, best_step = val0, 0

    for step in range(1, max_iters + 1):
        lr = learning_rate_at(step, max_iters)
        for g in optimizer.param_groups:
            g['lr'] = lr

        optimizer.zero_grad(set_to_none=True)
        xb, yb = BATCHER.sample('train')
        _, loss = model(xb, yb)
        loss.backward()
        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        if step % eval_interval == 0 or step == max_iters:
            val_loss = estimate_val_loss(model, eval_batches)
            train_loss = estimate_train_loss(model, eval_batches)
            if val_loss < best_val:
                best_val, best_step = val_loss, step
            history.append({'step': step, 'lr': lr,
                            'train_loss': train_loss, 'val_loss': val_loss})
            if verbose:
                print(f'{condition:16s} seed={seed} step={step:4d} '
                      f'lr={lr:.2e} train={train_loss:.4f} val={val_loss:.4f} '
                      f'best={best_val:.4f}@{best_step} elapsed={time.time()-started:.0f}s')

    return {
        'condition': condition,
        'seed': seed,
        'model_class': model_cls.__name__,
        'switch': asdict(switch),
        'best_step': best_step,
        'best_val_loss': best_val,
        'best_val_ppl': math.exp(min(best_val, 20)),
        'final_val_loss': history[-1]['val_loss'],
        'final_train_loss': history[-1]['train_loss'],
        'param_count': model.param_count(),
        'history': history,
    }


In [9]:
# Baseline residual conditions
# These definitions are retained for parameter checks/comparison only.
# Experiment 2 will run ONLY DiagonalOnly seed=1337 by default.
ATTN_ONLY = ResidualSwitch(True, False, 1.0, 1.0)
LOCAL_MIXER_ONLY = ResidualSwitch(True, False, 1.0, 1.0)
FFN_ONLY = ResidualSwitch(False, True, 1.0, 1.0)

set_seed(BASE_SEED)
m = ResidualGPT(ATTN_ONLY)
print(f"AttnOnly params: {m.param_count():,}")
del m


AttnOnly params: 10,750,080


In [10]:
# ============================================================
# Experiment 2: Diagonal-only attention
# ============================================================
# This preserves the SAME qkv/out parameterization as CausalSelfAttention.
# The only intervention is the attention mask:
#   token t may attend to t, and to no other position.
#
# We use PyTorch SDPA with an explicit boolean identity mask so that the
# attention machinery and dropout path remain as close as possible to the
# original CausalSelfAttention implementation.

class DiagonalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        assert N_EMBD % N_HEAD == 0
        self.qkv  = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)
        self.out  = nn.Linear(N_EMBD, N_EMBD, bias=False)
        self.drop = nn.Dropout(DROPOUT)
        self.n_head = N_HEAD
        self.head_dim = N_EMBD // N_HEAD

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # True = allowed. Only the diagonal is visible.
        diagonal_mask = torch.eye(T, device=x.device, dtype=torch.bool)

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=diagonal_mask,
            dropout_p=self.drop.p if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.out(y))


class DiagonalBlock(nn.Module):
    def __init__(self, switch: ResidualSwitch):
        super().__init__()
        self.switch = switch
        self.ln1    = nn.LayerNorm(N_EMBD)
        self.ln2    = nn.LayerNorm(N_EMBD)
        self.attn   = DiagonalSelfAttention()
        self.mlp    = MLP()

    def forward(self, x):
        a = self.attn(self.ln1(x))
        x = x + a if self.switch.use_attention_skip else a
        m = self.mlp(self.ln2(x))
        x = x + m if self.switch.use_mlp_skip else m
        return x


class DiagonalGPT(nn.Module):
    def __init__(self, switch: ResidualSwitch):
        super().__init__()
        self.switch = switch
        self.token_embed = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_embed   = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.drop        = nn.Dropout(DROPOUT)
        self.blocks      = nn.ModuleList([DiagonalBlock(switch) for _ in range(N_LAYER)])
        self.ln_f        = nn.LayerNorm(N_EMBD)
        self.lm_head     = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)

        # Same weight tying as ResidualGPT.
        self.lm_head.weight = self.token_embed.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def apply_output_gains(self):
        if self.switch.mlp_gain != 1.0:
            with torch.no_grad():
                for block in self.blocks:
                    block.mlp.fc2.weight.mul_(self.switch.mlp_gain)
        if self.switch.attention_gain != 1.0:
            with torch.no_grad():
                for block in self.blocks:
                    block.attn.out.weight.mul_(self.switch.attention_gain)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.token_embed(idx) + self.pos_embed(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def param_count(self):
        return sum(p.numel() for p in self.parameters())


In [11]:
# ============================================================
# Pre-run checks
# ============================================================
DIAGONAL_ONLY = ResidualSwitch(True, False, 1.0, 1.0)

set_seed(BASE_SEED)
attn_model = ResidualGPT(ATTN_ONLY)
set_seed(BASE_SEED)
diag_model = DiagonalGPT(DIAGONAL_ONLY)

print(f'AttnOnly params:     {attn_model.param_count():,}')
print(f'DiagonalOnly params: {diag_model.param_count():,}')
assert attn_model.param_count() == diag_model.param_count(), 'Parameter-count mismatch'

# Verify the intervention: changing another token must not change position t's
# diagonal-attention output when dropout is disabled via eval().
probe = DiagonalSelfAttention().to(DEVICE).eval()
x1 = torch.randn(1, 4, N_EMBD, device=DEVICE)
x2 = x1.clone()
x2[:, 0, :] += 10.0
with torch.no_grad():
    y1 = probe(x1)
    y2 = probe(x2)
# Positions 1..3 cannot see position 0.
max_cross_effect = (y1[:, 1:, :] - y2[:, 1:, :]).abs().max().item()
print('Max cross-position effect:', max_cross_effect)
assert max_cross_effect < 1e-6

print('Pre-run checks passed.')

del attn_model, diag_model, probe, x1, x2, y1, y2
if DEVICE == 'cuda':
    torch.cuda.empty_cache()


AttnOnly params:     10,750,080
DiagonalOnly params: 10,750,080
Max cross-position effect: 0.0
Pre-run checks passed.


In [12]:
# ============================================================
# FULL RUN: DiagonalOnly seed=1337
# ============================================================
# Run ONLY this seed first.

diagonal_1337 = train_one_run(
    condition='DiagonalOnly',
    seed=1337,
    model_cls=DiagonalGPT,
    switch=DIAGONAL_ONLY,
    max_iters=3000,
    eval_interval=100,
    eval_batches=EVAL_BATCHES,
    verbose=True,
)

save_result(diagonal_1337)

print('\nBest val:', diagonal_1337['best_val_loss'])
print('Best step:', diagonal_1337['best_step'])
print('Final val:', diagonal_1337['final_val_loss'])
print('Params:', diagonal_1337['param_count'])

print("\nMatched seed-1337 reference losses:")
print("AttnOnly:      ", MATCHED_ATTNONLY_SEED1337)
print("LocalMixerOnly:", MATCHED_LOCALMIXER_SEED1337)
print("FFNOnly:       ", MATCHED_FFNONLY_SEED1337)
print("DiagonalOnly:  ", diagonal_1337["best_val_loss"])


DiagonalOnly     seed=1337 step= 100 lr=3.00e-04 train=2.5653 val=2.5700 best=2.5700@100 elapsed=6s
DiagonalOnly     seed=1337 step= 200 lr=2.99e-04 train=2.5077 val=2.5102 best=2.5102@200 elapsed=11s
DiagonalOnly     seed=1337 step= 300 lr=2.97e-04 train=2.4891 val=2.5072 best=2.5072@300 elapsed=15s
DiagonalOnly     seed=1337 step= 400 lr=2.93e-04 train=2.4856 val=2.5043 best=2.5043@400 elapsed=20s
DiagonalOnly     seed=1337 step= 500 lr=2.88e-04 train=2.4746 val=2.4939 best=2.4939@500 elapsed=24s
DiagonalOnly     seed=1337 step= 600 lr=2.81e-04 train=2.4752 val=2.5031 best=2.4939@500 elapsed=29s
DiagonalOnly     seed=1337 step= 700 lr=2.72e-04 train=2.4717 val=2.5064 best=2.4939@500 elapsed=34s
DiagonalOnly     seed=1337 step= 800 lr=2.63e-04 train=2.4688 val=2.4993 best=2.4939@500 elapsed=38s
DiagonalOnly     seed=1337 step= 900 lr=2.52e-04 train=2.4706 val=2.4984 best=2.4939@500 elapsed=43s
DiagonalOnly     seed=1337 step=1000 lr=2.41e-04 train=2.4704 val=2.4984 best=2.4939@500 ela

In [13]:
# Optional: package this experiment directory before terminating the instance.
import shutil
archive = shutil.make_archive(
    '/workspace/10m_diagonal_attention_experiment_backup',
    'zip',
    OUT_DIR,
)
print('Created:', archive)


Created: /workspace/10m_diagonal_attention_experiment_backup.zip


In [14]:
# ============================================================
# EXPERIMENT 2 FOLLOW-UP
# DiagonalOnly — seeds 42 and 123
# ============================================================

diagonal_results = {}

for seed in [42, 123]:
    print("\n" + "=" * 72)
    print(f"FULL RUN: DiagonalOnly seed={seed}")
    print("=" * 72)

    result = train_one_run(
        condition="DiagonalOnly",
        seed=seed,
        model_cls=DiagonalGPT,
        switch=DIAGONAL_ONLY,
        max_iters=3000,
        eval_interval=100,
        eval_batches=EVAL_BATCHES,
        verbose=True
    )

    # Keep result in memory
    diagonal_results[seed] = result

    # IMPORTANT: save immediately
    save_result(result)

    print(f"\nSeed {seed} complete")
    print("Best val:", result["best_val_loss"])
    print("Best step:", result["best_step"])
    print("Final val:", result["final_val_loss"])
    print("Params:", result["param_count"])


FULL RUN: DiagonalOnly seed=42
DiagonalOnly     seed=42 step= 100 lr=3.00e-04 train=2.5531 val=2.5546 best=2.5546@100 elapsed=6s
DiagonalOnly     seed=42 step= 200 lr=2.99e-04 train=2.5007 val=2.5209 best=2.5209@200 elapsed=10s
DiagonalOnly     seed=42 step= 300 lr=2.97e-04 train=2.4888 val=2.5104 best=2.5104@300 elapsed=15s
DiagonalOnly     seed=42 step= 400 lr=2.93e-04 train=2.4824 val=2.5027 best=2.5027@400 elapsed=19s
DiagonalOnly     seed=42 step= 500 lr=2.88e-04 train=2.4773 val=2.4959 best=2.4959@500 elapsed=24s
DiagonalOnly     seed=42 step= 600 lr=2.81e-04 train=2.4779 val=2.4996 best=2.4959@500 elapsed=29s
DiagonalOnly     seed=42 step= 700 lr=2.72e-04 train=2.4720 val=2.4961 best=2.4959@500 elapsed=33s
DiagonalOnly     seed=42 step= 800 lr=2.63e-04 train=2.4700 val=2.4997 best=2.4959@500 elapsed=38s
DiagonalOnly     seed=42 step= 900 lr=2.52e-04 train=2.4709 val=2.5053 best=2.4959@500 elapsed=42s
DiagonalOnly     seed=42 step=1000 lr=2.41e-04 train=2.4699 val=2.4907 best=2.

In [15]:
import numpy as np

vals = [
    diagonal_1337["best_val_loss"],
    diagonal_results[42]["best_val_loss"],
    diagonal_results[123]["best_val_loss"],
]

print("=" * 72)
print("DIAGONAL-ONLY SUMMARY")
print("=" * 72)

print(f"Seed 1337: {vals[0]:.6f}")
print(f"Seed   42: {vals[1]:.6f}")
print(f"Seed  123: {vals[2]:.6f}")

print()
print(f"Mean:       {np.mean(vals):.6f}")
print(f"Sample std: {np.std(vals, ddof=1):.6f}")
print(f"Min:        {np.min(vals):.6f}")
print(f"Max:        {np.max(vals):.6f}")

DIAGONAL-ONLY SUMMARY
Seed 1337: 2.490095
Seed   42: 2.490724
Seed  123: 2.491776

Mean:       2.490865
Sample std: 0.000849
Min:        2.490095
Max:        2.491776


In [16]:
for f in sorted(CKPT_DIR.glob("*.json")):
    print(f.name)

DiagonalOnly_seed123.json
DiagonalOnly_seed1337.json
DiagonalOnly_seed42.json


In [17]:
import shutil
import os

SOURCE_DIR = "/workspace/10m_diagonal_attention_mechanism_test"
ZIP_PATH = "/workspace/experiment"

shutil.make_archive(
    ZIP_PATH,
    "zip",
    SOURCE_DIR
)

print("Created:", ZIP_PATH + ".zip")
print("Size:", round(os.path.getsize(ZIP_PATH + ".zip") / 1024**2, 2), "MB")

Created: /workspace/experiment.zip
Size: 0.42 MB


In [18]:
import zipfile

with zipfile.ZipFile("/workspace/experiment.zip", "r") as z:
    for name in z.namelist():
        print(name)

data/
results/
results/DiagonalOnly_seed1337.json
results/DiagonalOnly_seed42.json
results/DiagonalOnly_seed123.json
data/input.txt
